# 04 — Stat-only likelihood scans

Use the [NSBI LHC toolkit](https://github.com/iris-hep/nsbi-lhc-toolkit/) for an **unbinned, extended likelihood**: both the event yield and the density-ratio information enter the fit. The only fitted parameter is $\mu$; there are no nuisance parameters.

| Process | Scaling |
|---|---|
| SBI | $\sqrt{\mu}$ |
| S | $\mu-\sqrt{\mu}$ |
| B | $1-\sqrt{\mu}$ |
| qqZZ | $1$ |

The reference is still **S + EWSBI10**, with signed `weight_kFact` weights. Use the **full reference first**, then **2,048**, **16,384**, **131,072**, and **1,048,576** randomly selected events. One seeded permutation makes the smaller subsets nested, with no replacement.

We use **SM Asimov data at $\mu=1$** on each reference sample. All four physical process yields come from their full native samples and stay fixed for every curve. Reducing the reference size changes the Monte Carlo approximation, **not the luminosity or expected event yield**. These are expected likelihood curves, not fits to observed collision data.

Also compare the full-reference reweighted Asimov with a **direct SBI + qqZZ Asimov**, using the native events and their original weights. Both curves use the same full-reference-normalized likelihood model. The direct curve is a closure test and its fitted minimum is not forced to mu = 1.


## Setup

Use a **CPU runtime**. The toolkit is pinned to an inspected commit, and the Matplotlib/mplhep versions match notebooks 01–03. The CPU-only setup removes unused CUDA JAX plugins and installs matching JAX/JAXlib versions. This fixes the separate Colab plugin-version messages seen in the previous run. If setup reports a stale import, choose **Runtime → Restart session**, then **Run all**.

`utils.py` supplies the existing HDF5 reader. `utils_likelihood.py` formats the toolkit workspace, checks the physical scan domain, and maps the POI to its normalization factors. It contains no likelihood implementation. The toolkit handles the likelihood, fit, scan, and curve drawing.

In [ ]:
# This notebook uses CPU JAX. Remove only installed CUDA JAX plugin packages.
import sys
import subprocess
from importlib.metadata import distributions
cuda_names = {"jax-cuda12-plugin", "jax-cuda12-pjrt", "jax-cuda13-plugin", "jax-cuda13-pjrt"}
installed_cuda = [dist.metadata["Name"] for dist in distributions()
                  if dist.metadata.get("Name", "").lower() in cuda_names]
if installed_cuda:
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", *installed_cuda])

%pip -q install "matplotlib==3.10.0" "mplhep==0.4.1" tables "jax[cpu]==0.6.2" "iminuit==2.33.0" "scikit-learn==1.9.1" "git+https://github.com/iris-hep/nsbi-lhc-toolkit.git@fc09848fc6540fd32310faebbe9db6eea7ecd17b"


In [ ]:



import sys
import os
from importlib.metadata import version

packages = {name: name for name in (
    "numpy", "pandas", "matplotlib", "mplhep", "tables", "jax", "jaxlib", "iminuit", "scipy"
)}
packages["sklearn"] = "scikit-learn"
for module_name, package in packages.items():
    loaded = sys.modules.get(module_name)
    if loaded is not None and loaded.__version__ != version(package):
        raise RuntimeError(
            f"{package} changed on disk while it was loaded. "
            "Choose Runtime > Restart session, then Run all."
        )
os.environ["JAX_PLATFORMS"] = "cpu"

import gc
import json
import importlib
from pathlib import Path
from urllib.request import urlretrieve
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mplhep as hep
import jax
from IPython.display import display
from nsbi_common_utils import models, inference

try:
    from google.colab import drive
except ImportError:
    import utils as offshell_utils
    import utils_likelihood
else:
    drive.mount("/content/drive")
    repo_url = "https://raw.githubusercontent.com/rafaellopesdesa/offshell_reweighting/main"
    urlretrieve(f"{repo_url}/utils.py", "/content/offshell_utils.py")
    urlretrieve(f"{repo_url}/utils_likelihood.py", "/content/utils_likelihood.py")
    import offshell_utils
    import utils_likelihood

importlib.invalidate_caches()
importlib.reload(offshell_utils)
importlib.reload(utils_likelihood)
hep.style.use("ATLAS")
offshell_utils.check_plot_exports()
print("Toolkit commit:", utils_likelihood.TOOLKIT_COMMIT)
print("JAX devices:", jax.devices())
print("PNG and PDF export checks passed.")


In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/HZZ_OffShell")
DATA_PATH = PROJECT_DIR / "dataset_wDR_mean_origIndex.h5"
RESULT_DIR = PROJECT_DIR / "likelihoods" / "04_stat_only"
PLOT_DIR = PROJECT_DIR / "plots" / "04_likelihood_scans"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

PROCESSES = ["SBI", "S", "B", "qqZZ"]
SUBSET_SIZES = [2048, 16384, 131072, 1048576]
RANDOM_SEED = 42
SCAN_RANGE = (0.0, 1.8)
SCAN_POINTS = 73  # Step 0.025; includes the Asimov truth mu=1 exactly.


## Load the reference and fix the physical yields

Keep the same finite-value selection as notebooks 01–03: finite `weight_kFact`, `NN_disc_New`, and `m4l_fsr`. Values outside the old histogram ranges remain included. The likelihood uses the stored ratios directly, without binning those variables.

Read ratios on both reference and direct SBI + qqZZ events; both are used to evaluate the likelihood. All signed weights are retained. The native S sample also belongs to the reference; that overlap is intentional.

Use **`Type == "qqZZ"` and `subType != "ttV"`** for the native qqZZ yield and the direct SBI + qqZZ Asimov. ttV is excluded throughout this notebook.

In [ ]:
data = offshell_utils.load_columns(DATA_PATH, [
    "Type", "subType", "weight_kFact", "NN_disc_New", "m4l_fsr",
    "r_SBI", "r_S", "r_B", "r_qqZZ",
])
event_type = data["Type"]
is_qqzz = (event_type == "qqZZ") & (data["subType"] != "ttV")
weights = np.asarray(data["weight_kFact"], dtype=np.float64)
finite = (np.isfinite(weights) & np.isfinite(data["NN_disc_New"])
          & np.isfinite(data["m4l_fsr"]))
reference = np.isin(event_type, ["S", "EWSBI10"]) & finite
reference_rows = np.flatnonzero(reference)
w_ref = weights[reference]
r_ref = {name: np.asarray(data[f"r_{name}"][reference], dtype=np.float64)
         for name in PROCESSES}
direct = ((event_type == "SBI") | is_qqzz) & finite
direct_rows = np.flatnonzero(direct)
w_direct = weights[direct]
r_direct = {name: np.asarray(data[f"r_{name}"][direct], dtype=np.float64)
            for name in PROCESSES}
for sample_name, ratios_by_process in [("reference", r_ref), ("direct", r_direct)]:
    for name, ratios in ratios_by_process.items():
        if not np.isfinite(ratios).all():
            raise ValueError(f"Nonfinite r_{name} on {sample_name} events; inspect the input.")

native_yields = {}
yield_rows = []
for name in ["Reference"] + PROCESSES:
    if name == "Reference":
        original = np.isin(event_type, ["S", "EWSBI10"])
    elif name == "qqZZ":
        original = is_qqzz
    else:
        original = event_type == name
    mask = original & finite
    total = float(weights[mask].sum())
    native_yields[name] = total
    yield_rows.append({"sample": name, "events": int(mask.sum()), "yield": total,
                       "excluded_nonfinite": int(np.count_nonzero(original & ~finite)),
                       "negative_weights": int(np.count_nonzero(weights[mask] < 0))})
yield_table = pd.DataFrame(yield_rows).set_index("sample")
display(yield_table)
yield_table.to_csv(RESULT_DIR / "native_yields.csv")
if any(not np.isfinite(value) or value <= 0 for value in native_yields.values()):
    raise ValueError("Each physical sample needs a finite, positive total signed yield.")
if max(SUBSET_SIZES) > len(w_ref):
    raise ValueError(f"Only {len(w_ref):,} selected reference events; cannot draw all sizes.")

# Keep only the arrays used to prepare the two Asimov constructions.
del data, event_type, is_qqzz, weights, finite, reference, direct, original, mask, ratios, ratios_by_process
gc.collect()


## Same-sample normalization for every reference size

For the selected reference events, calculate
$$
u_i = \frac{w_i}{\sum_{k\in\mathrm{subset}} w_k},\qquad
Z_j = \sum_{i\in\mathrm{subset}} u_i r_j(x_i),\qquad
\widehat r_j(x_i)=\frac{r_j(x_i)}{Z_j}.
$$
Do this separately for **SBI, S, B, and qqZZ**, including the full-reference case. Each corrected weighted mean must equal one. Never reuse the full-sample Z factors for a smaller sample.

At $\mu=1$, only SBI and qqZZ contribute. Supply the toolkit with the Asimov event weights
$$
a_i = u_i\left(\lambda_{\mathrm{SBI}}\widehat r_{\mathrm{SBI}}(x_i)
+\lambda_{\mathrm{qqZZ}}\widehat r_{\mathrm{qqZZ}}(x_i)\right).
$$
Their sum is always $\lambda_{\mathrm{SBI}}+\lambda_{\mathrm{qqZZ}}$. Negative MC weights remain signed. The resulting curves probe reference-sample convergence; there are no MC-statistical or ratio-training nuisance parameters.

In [ ]:
permutation = np.random.default_rng(RANDOM_SEED).permutation(len(w_ref))
reference_cases = [("full", len(w_ref))] + [(f"n_{n}", n) for n in SUBSET_SIZES]
workspace_paths = {}
normalization_rows = []
expected_yield = native_yields["SBI"] + native_yields["qqZZ"]

for label, n in reference_cases:
    selected = np.arange(n) if label == "full" else permutation[:n]
    sample_dir = RESULT_DIR / label
    sample_dir.mkdir(parents=True, exist_ok=True)
    selected_weight_sum = float(w_ref[selected].sum())
    if not np.isfinite(selected_weight_sum) or selected_weight_sum == 0:
        raise ValueError(f"{label}: reference weights cannot be normalized.")
    u = w_ref[selected] / selected_weight_sum
    np.testing.assert_allclose(u.sum(), 1.0, rtol=1e-10)

    z = {}
    corrected = {}
    for name in PROCESSES:
        r = r_ref[name][selected]
        z[name] = float(np.sum(u * r))
        if not np.isfinite(z[name]) or z[name] == 0:
            raise ValueError(f"{label}: invalid Z_{name} = {z[name]}")
        corrected[name] = r / z[name]
        mean_after = float(np.sum(u * corrected[name]))
        np.testing.assert_allclose(mean_after, 1.0, rtol=1e-10)
        np.save(sample_dir / f"r_{name}.npy", corrected[name])
        normalization_rows.append({"sample": label, "events": n, "process": name,
                                   "Z": z[name], "corrected_mean": mean_after})

    if label == "full":
        full_z = z.copy()

    asimov = u * (native_yields["SBI"] * corrected["SBI"]
                  + native_yields["qqZZ"] * corrected["qqZZ"])
    np.testing.assert_allclose(asimov.sum(), expected_yield, rtol=1e-10)
    np.save(sample_dir / "asimov_weights.npy", asimov)
    np.save(sample_dir / "reference_weights_normalized.npy", u)
    np.save(sample_dir / "event_rows.npy", reference_rows[selected])
    workspace_paths[label] = utils_likelihood.write_workspace(
        sample_dir, native_yields,
        metadata={"reference_events": n, "reference_types": ["S", "EWSBI10"],
                  "qqZZ_selection": "(Type == 'qqZZ') & (subType != 'ttV')",
                  "asimov_source": "reweighted_reference", "asimov_events": n,
                  "normalization_reference": label, "scan_range": list(SCAN_RANGE),
                  "random_seed": RANDOM_SEED, "Z": z,
                  "selected_reference_weight_sum": selected_weight_sum,
                  "asimov_yield": float(asimov.sum()),
                  "negative_reference_weights": int(np.count_nonzero(u < 0)),
                  "input_file": str(DATA_PATH.resolve())},
    )
    print(f"{label}: {n:,} reference events; Asimov yield = {asimov.sum():.6f}")

normalization_table = pd.DataFrame(normalization_rows)
display(normalization_table.set_index(["sample", "process"]))
normalization_table.to_csv(RESULT_DIR / "normalization_checks.csv", index=False)
del corrected, r, asimov, u, selected
gc.collect()


## Direct SBI + qqZZ Asimov

For these events, use the original `weight_kFact` as the Asimov weights. Do **not** reweight the direct events again.

Only the Asimov sample changes. Evaluate the model ratios on these direct events using the **same full-reference Z factors** as the full-reference curve: $\widehat r_j(x)=r_j(x)/Z_j^{\rm full}$. The normalizing measure is still the reference, so averaging the ratios over SBI + qqZZ would normalize a different model and spoil the comparison.

The total direct Asimov weight must equal the same native SBI + qqZZ yield. Its likelihood minimum can differ from mu = 1 because of finite MC statistics or imperfect density ratios.

In [ ]:
direct_dir = RESULT_DIR / "direct"
direct_dir.mkdir(parents=True, exist_ok=True)
for name in PROCESSES:
    np.save(direct_dir / f"r_{name}.npy", r_direct[name] / full_z[name])
np.testing.assert_allclose(w_direct.sum(), expected_yield, rtol=1e-10)
np.save(direct_dir / "asimov_weights.npy", w_direct)
np.save(direct_dir / "event_rows.npy", direct_rows)
workspace_paths["direct"] = utils_likelihood.write_workspace(
    direct_dir, native_yields,
    metadata={"reference_events": len(w_ref), "reference_types": ["S", "EWSBI10"],
              "qqZZ_selection": "(Type == 'qqZZ') & (subType != 'ttV')",
              "asimov_source": "native_SBI_qqZZ", "asimov_events": len(w_direct),
              "normalization_reference": "full", "Z": full_z,
              "scan_range": list(SCAN_RANGE), "asimov_yield": float(w_direct.sum()),
              "negative_asimov_weights": int(np.count_nonzero(w_direct < 0)),
              "input_file": str(DATA_PATH.resolve())},
)
cases = reference_cases + [("direct", len(w_direct))]
print(f"Direct SBI + qqZZ: {len(w_direct):,} events; yield = {w_direct.sum():.6f}")
del r_direct, w_direct, direct_rows
gc.collect()


## Fit and scan with the toolkit

Each `workspace.json` stores the four full process yields, paths to the corrected ratios, and the Asimov weights. Its metadata records the subset, seed, Z factors, and physical scaling formula.

The pinned toolkit supports multiplicative normfactors. The small `mu_likelihood` adapter ties its three internal factors to $(\sqrt\mu,\mu-\sqrt\mu,1-\sqrt\mu)$; **only one physical parameter is fitted**. qqZZ has no modifier. Negative S or B coefficients are allowed; the combined intensity must remain physical. No densities or weights are clipped.

`models.sbi_parametric_model` computes the extended likelihood. `inference.inference` fits and scans it. The model has no nuisance parameters, so the ordinary scan is already **stat-only**. The toolkit returns $t_\mu=-2\Delta\log L$ with the scan minimum subtracted. The grid contains the Asimov truth $\mu=1$ exactly.

With the earlier ttV-inclusive qqZZ yield, the full-reference scan became undefined above mu ≈ 1.8947: the combined intensity of an event crossed zero. Same-sample normalization enforces the integrals but does not guarantee positive mixtures when interference coefficients subtract. The default range is **[0, 1.8]**. Recheck the domain after applying the corrected qqZZ yield; the earlier zero is not a bound for the updated model.

Before fitting, check positivity over the **whole interval**, not just the scan grid. In $t=\sqrt\mu$, the total intensity is quadratic, so the check covers the endpoints and any interior minimum. It saves `domain_check.json`, including the worst event's input row. This is done for every reference subset and the direct sample. The parameter adapter also keeps the fit inside the checked interval. Negative weights and negative individual process coefficients remain unchanged.


In [ ]:
scan_grid = np.linspace(*SCAN_RANGE, SCAN_POINTS)
if SCAN_RANGE[0] < 0 or not np.any(np.isclose(scan_grid, 1.0, atol=1e-12, rtol=0)):
    raise ValueError("Use a nonnegative scan grid containing mu=1 exactly.")

scans = {}
fit_rows = []
domain_rows = []
for label, n in cases:
    print(f"\n--- {label}: {n:,} Asimov events ---")
    domain = utils_likelihood.check_workspace_domain(workspace_paths[label], SCAN_RANGE)
    domain_rows.append({"sample": label, **domain})
    print(f"Minimum intensity over the full interval: {domain['min_intensity']:.6g}")
    workspace = json.loads(workspace_paths[label].read_text())
    model = models.sbi_parametric_model(workspace, measurement_to_fit="stat_only")
    nll_mu = utils_likelihood.mu_likelihood(model, mu_bounds=SCAN_RANGE)
    if not np.isfinite(nll_mu([1.0])):
        raise ValueError(f"{label}: nonphysical total intensity at the Asimov point.")

    fitter = inference.inference(
        model_nll=nll_mu, initial_values=[1.0], list_parameters=["mu"],
        num_unconstrained_params=1,
    )
    fitter.perform_fit()
    mu, t_mu = fitter.perform_profile_scan(
        parameter_name="mu", bound_range=SCAN_RANGE, size=SCAN_POINTS,
    )
    mu, t_mu = np.asarray(mu), np.asarray(t_mu)
    if not np.isfinite(t_mu).all():
        raise ValueError(
            f"{label}: nonfinite likelihood scan. Inspect the signed ratios and "
            "combined intensity over SCAN_RANGE; no invalid points were clipped."
        )
    scans[label] = (mu, t_mu)
    pd.DataFrame({"mu": mu, "t_mu": t_mu}).to_csv(
        workspace_paths[label].parent / "scan.csv", index=False,
    )
    fit_rows.append({"sample": label, "asimov_events": n,
                     "reference_events": workspace["metadata"]["reference_events"],
                     "mu_hat": float(fitter.pulls_global_fit[0]),
                     "grid_minimum_mu": float(mu[np.argmin(t_mu)]),
                     "asimov_yield": workspace["metadata"]["asimov_yield"]})
    del fitter, nll_mu, model
    jax.clear_caches()  # Release the large compiled model before the next subset.
    gc.collect()

domain_table = pd.DataFrame(domain_rows).set_index("sample")
display(domain_table[["min_intensity", "mu_at_min_intensity", "nonpositive_events", "min_total_yield"]])
domain_table.to_csv(RESULT_DIR / "domain_checks.csv")
fit_table = pd.DataFrame(fit_rows).set_index("sample")
display(fit_table)
fit_table.to_csv(RESULT_DIR / "fit_summary.csv")


## Compare reference sizes and Asimov constructions

The first figure compares the five reference sizes. The second compares the **full-reference reweighted Asimov** with the **direct SBI + qqZZ Asimov** using the same model. Each curve is relative to its own scan minimum.

The toolkit draws the curves; the helper applies ATLAS formatting and scales the likelihood axis to the plotted values. These are likelihood axes, not histogram ratio panels.

In [ ]:
plot_groups = [
    ("reference_size_comparison", [label for label, _ in reference_cases],
     [f"Full reference ({len(w_ref):,})"] + [f"{n:,} reference events" for n in SUBSET_SIZES]),
    ("asimov_comparison", ["full", "direct"],
     ["Full reference reweighted", "Direct SBI + qqZZ"]),
]
for filename, keys, labels in plot_groups:
    fig, ax = utils_likelihood.plot_likelihood_scans(
        [scans[key] for key in keys], labels, SCAN_RANGE,
    )
    for extension in ("png", "pdf"):
        fig.savefig(PLOT_DIR / f"{filename}.{extension}", dpi=180, bbox_inches="tight")
    plt.show()
print("Workspaces, arrays, Z checks, domain checks, and scan tables:", RESULT_DIR)
print("Likelihood figures:", PLOT_DIR)


The Z table should have `corrected_mean = 1` for all 20 reference entries. All six Asimov yields should agree. The five reweighted-reference minima should be at or close to mu = 1; the **direct SBI + qqZZ minimum is not forced to one**. Comparing it with the full-reference curve tests closure of the ratio model against the native samples.

The JSON workspaces and their `.npy` inputs stay in Drive. To reopen one, load its JSON, create `models.sbi_parametric_model(workspace, "stat_only")`, and apply `utils_likelihood.mu_likelihood(model)` before fitting the single physical parameter. The adapter reads the saved scan range. The three internal normalization factors must stay tied to mu.